In [47]:
import pandas as pd
import numpy as np
import os

from PIL import Image
from matplotlib.image import imread
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, silhouette_score
from sklearn.utils.class_weight import compute_class_weight


from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import pickle



In [34]:
X = np.load(os.path.join("datos_procesados/X_bn.npy"))

y = np.load(os.path.join("datos_procesados/y_bn.npy"))

In [35]:
X.shape

(28821, 48, 48, 1)

In [36]:
y.shape

(28821, 7)

In [37]:
np.max(X)

np.float32(255.0)

In [38]:
np.min(X)

np.float32(0.0)

In [39]:
X_st = X / 255.0
print(np.max(X_st), np.min(X_st))

1.0 0.0


In [40]:
y_labels = np.argmax(y, axis=1) if len(y.shape) > 1 else y

In [41]:
X_train, X_val, y_train, y_val = train_test_split(
    X_st, y,
    test_size=0.1,       
    stratify=y_labels,   
    random_state=11
)

X_train, y_train = shuffle(X_train, y_train, random_state=11)

print("Tamaño X_train:", X_train.shape)
print("Tamaño X_val:", X_val.shape)
print("Tamaño X_test:", y_train.shape)
print("Tamaño y_val:", y_val.shape)

Tamaño X_train: (25938, 48, 48, 1)
Tamaño X_val: (2883, 48, 48, 1)
Tamaño X_test: (25938, 7)
Tamaño y_val: (2883, 7)


In [42]:
y_train_labels = np.argmax(y_train, axis=1)
classes = np.unique(y_train_labels)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_labels
)
class_weights = dict(zip(classes, class_weights))

In [43]:
early_stop = EarlyStopping(
    monitor='val_categorical_accuracy',
    patience=10, # Más paciencia para ver el efecto del LR reducer
    restore_best_weights=True,
    verbose=1
)

In [44]:
lr_reducer = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

In [45]:
datagen = ImageDataGenerator(
    rotation_range=15,      
    width_shift_range=0.1,   
    height_shift_range=0.1,  
    zoom_range=0.1,         
    horizontal_flip=True,    
    fill_mode='nearest'      
)

train_generator = datagen.flow(X_train, y_train, batch_size=32)  

In [ ]:
layers = [
    # Captura de bordes y texturas simples
    keras.layers.Conv2D(64, (3,3), padding='same', input_shape=(48,48,1)),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Conv2D(64, (3,3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Dropout(0.25),

    # Captura de formas
    keras.layers.Conv2D(128, (3,3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Conv2D(128, (3,3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Dropout(0.3),

    # Patrones complejos de expresión
    keras.layers.Conv2D(256, (3,3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Conv2D(256, (3,3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Dropout(0.4),


    keras.layers.Flatten(),
    keras.layers.Dense(512),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dropout(0.5),
    
    keras.layers.Dense(7, activation='softmax')
]

model_cnn = keras.Sequential(layers)

# 2. Compilación con Learning Rate inicial ligeramente más bajo
model_cnn.compile(
    optimizer=Adam(learning_rate=0.0005), 
    loss='categorical_crossentropy',
    metrics=[keras.metrics.CategoricalAccuracy()]
)

In [49]:
history_cnn = model_cnn.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=100, 
    class_weight=class_weights,
    callbacks=[early_stop, lr_reducer]
)

Epoch 1/100
811/811 ━━━━━━━━━━━━━━━━━━━━ 671s 820ms/step - categorical_accuracy: 0.1921 - loss: 2.2031 - val_categorical_accuracy: 0.2171 - val_loss: 1.8596 - learning_rate: 5.0000e-04
Epoch 2/100
811/811 ━━━━━━━━━━━━━━━━━━━━ 687s 847ms/step - categorical_accuracy: 0.2417 - loss: 1.9385 - val_categorical_accuracy: 0.3417 - val_loss: 1.7397 - learning_rate: 5.0000e-04
Epoch 3/100
811/811 ━━━━━━━━━━━━━━━━━━━━ 3996s 5s/step - categorical_accuracy: 0.3009 - loss: 1.7734 - val_categorical_accuracy: 0.3698 - val_loss: 1.6337 - learning_rate: 5.0000e-04
Epoch 4/100
811/811 ━━━━━━━━━━━━━━━━━━━━ 581s 716ms/step - categorical_accuracy: 0.3636 - loss: 1.6425 - val_categorical_accuracy: 0.4096 - val_loss: 1.5365 - learning_rate: 5.0000e-04
Epoch 5/100
811/811 ━━━━━━━━━━━━━━━━━━━━ 640s 790ms/step - categorical_accuracy: 0.4145 - loss: 1.5180 - val_categorical_accuracy: 0.4520 - val_loss: 1.4463 - learning_rate: 5.0000e-04
Epoch 6/100
811/811 ━━━━━━━━━━━━━━━━━━━━ 1481s 2s/step - categorical_accuracy

In [51]:
model_cnn.save("modelos/model_cnn_ok.keras")